# LLMs & Generative AI — Session 2: Limitations, Responsible Use & Workflows
### ITI · Instructor: Ahmed Abdelsalam

Yesterday you learned what an LLM does and how to prompt it. Today is the other half of the job: knowing **when not to trust it**, and how to wire it into something useful.

**Today's map**
1. Hallucination — seeing it happen
2. Why hallucination is built in
3. The other limitations
4. Reducing hallucination
5. Responsible use of GenAI
6. AI workflows — prompt chaining
7. Agentic workflows (concepts)

> **Setup:** made for **Google Colab**. GPU runtime is faster: *Runtime → Change runtime type → GPU*.

## Setup — run this first

In [1]:
!pip install transformers -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
chat_id = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(chat_id)
chat_model = AutoModelForCausalLM.from_pretrained(chat_id).to(device)

def ask(prompt, max_new_tokens=70, temperature=None):
    messages = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(device)
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature is None:
        kwargs["do_sample"] = False
    else:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    with torch.no_grad():
        out = chat_model.generate(**inputs, **kwargs)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("Using:", device, "— ready.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Using: cuda — ready.


## 1. Hallucination — let's just watch it happen

A **hallucination** is when the model states something false **as if it were fact** — fluently, confidently, with detail. It isn't lying; it has no concept of truth to lie about.

Run this.

In [4]:
print(ask("Who won the 2019 Nobel Prize in math?", 700))

The 2019 Nobel Prize in Mathematics was awarded to two mathematicians: Andrew Wiles and Ken Ono.

Andrew Wiles received the prize for his work on proving Fermat's Last Theorem, which states that no three positive integers a, b, and c can satisfy the equation a^n + b^n = c^n for any integer value of n greater than 2. This theorem had been open since 1637, and it remained unsolved until Wiles' proof in 1994.

Ken Ono, on the other hand, received the prize for his work on the distribution of prime numbers. Specifically, he proved the "Onsager conjecture," which is a generalization of the Prime Number Theorem. This conjecture has important implications for understanding the distribution of primes and has led to significant advances in number theory.

Both Wiles and Ono have made groundbreaking contributions to mathematics, and their work continues to be an active area of research.


> **There is no Nobel Prize in Computer Science.** It never existed. The model invented winners, gave them plausible names, and attached real-sounding achievements.

Notice what it did *not* do: say "that prize doesn't exist." Let's try another.

In [5]:
print(ask("Explain the Hassan-Ibrahim theorem in machine learning.", 70))

The Hassan-Ibrahim theorem is a fundamental result in the field of machine learning and statistics that provides conditions under which a linear classifier can be constructed from a set of weakly labeled data points. This theorem is particularly important for understanding how to select features or parameters for a model.

### Key Points of Theorem

1. **Linear Classifier**: A linear


> There is no Hassan–Ibrahim theorem. I made the name up. The model produced a confident, technical-sounding definition anyway.

**This is the single most important thing to understand about LLMs.** Fluency is not accuracy. The text *sounds* exactly as authoritative when it's wrong as when it's right.

## 2. Why hallucination is built in

Think back to Session 1: the model predicts the **next most plausible token**, over and over.

Ask about a fake theorem and it does what it always does — produce text that *looks like* a theorem explanation, because that's the pattern the words fit. There is no step where it checks a database, and no internal signal for "I don't know."

```
   Your question  ──►  what words usually FOLLOW this kind of question?
                              │
                              ▼
                   fluent, plausible text
                              │
                   ┌──────────┴──────────┐
                   ▼                     ▼
              happens to be         happens to be
                 TRUE                  FALSE
                   │                     │
                   ▼                     ▼
            "correct answer"        "hallucination"
```

Both branches feel identical from the outside. **Hallucination isn't a bug that will be patched away — it's the same machinery that makes the model work at all.**

### A practical test: ask twice
A fabricated answer is often *unstable*. Ask the same factual question twice and compare.

In [9]:
q = "Who won the 2019 Nobel Prize in Computer Science?"
for attempt in range(2):
    print(f"--- attempt {attempt+1} ---")
    print(ask(q, 45, temperature=0.2))
    print()

--- attempt 1 ---
The 2019 Nobel Prize in Computer Science was awarded to two researchers: Andrew Ng and Geoffrey Hinton.

Andrew Ng, an artificial intelligence professor at the University of California, Berkeley, received the award for his work

--- attempt 2 ---
The 2019 Nobel Prize in Computer Science was awarded to two researchers: Peter Shor and Richard Mayers.

Peter Shor is known for his work on quantum error correction codes, which have applications in cryptography and



> If the two answers disagree on the *facts*, the model is almost certainly making them up. Real knowledge tends to be stable; invention wanders. This is a quick sanity check you can teach anyone.

In [ ]:
llm  !=  agent

## 3. The other limitations

### Counting and spelling
The model sees **tokens**, not letters — so letter-level questions are guesswork.

In [11]:
print("How many r's in 'strawberry'?  ->", ask("How many times does the letter r appear in the word strawberry? Answer with just the number.", 20))
print("(correct answer: 3)")

How many r's in 'strawberry'?  -> 2
(correct answer: 3)


### Arithmetic
It predicts what a calculation *looks like*, it doesn't calculate.

In [24]:
print("17 x 24 =", ask("What is 17 multiplied by 24? Answer with just the number.", 20))
print("(correct answer: 408)")

17 x 24 = 17 * 24 = 388
(correct answer: 408)


In [26]:
print(ask("what's today's date"))

Today's date is March 14, 2023.


### Knowledge cutoff
Training stopped on a date. Anything after that simply isn't in there — and the model may not realise it.

### Context window
There's a maximum number of tokens it can hold. Long documents must be split, and details in the middle of a long context often get missed.

### Non-determinism
With sampling on, the same prompt gives different answers. Great for creativity; a problem when you need reproducibility.

### Bias
It learned from human text, so it reproduces the patterns in that text — including stereotypes and skewed representation.

| Limitation | Practical consequence |
|---|---|
| Hallucination | never trust unverified facts |
| Tokens, not letters | bad at counting, spelling, anagrams |
| No arithmetic engine | use a calculator/code for maths |
| Knowledge cutoff | recent events unreliable |
| Context window | long inputs must be chunked |
| Non-determinism | outputs vary run to run |
| Bias | audit outputs affecting people |

## 4. Reducing hallucination

You can't eliminate it, but you can make it much less likely.

**1. Ground it — supply the facts in the prompt.** By far the most effective move. Instead of asking the model to recall, give it the source text and tell it to answer *only* from that.

In [30]:
context = """The ITI Level 2 Summer Training programme in Alexandria covers
Python, Computer Vision, YOLO object detection, and Generative AI. The Computer
Vision course runs for 12 hours split into lectures and labs."""

grounded = f"""Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly: "Not in the provided context."

Context:
{context}

Question: How many hours is the Computer Vision course?"""

print("Answerable  ->", ask(grounded, 40))
print()

missing = grounded.replace("How many hours is the Computer Vision course?",
                           "Who is the head of the ITI Alexandria branch?")
print("Unanswerable ->", ask(missing, 40))

Answerable  -> The Computer Vision course runs for 12 hours.

Unanswerable -> Not in the provided context.


> That second answer is the behaviour you want: **admitting the gap instead of inventing**. Grounding is the core idea behind **RAG** (Retrieval-Augmented Generation) — fetch relevant documents first, then ask the model to answer from them. That's how most production question-answering systems are built.

**2. Ask for uncertainty.** "If you are not sure, say so" measurably helps.
**3. Ask for sources** — then actually check them. Fabricated citations are common.
**4. Use the right tool.** Maths → a calculator. Current events → search. Don't ask the model to be something it isn't.
**5. Keep a human in the loop** for anything that matters.

## 5. Responsible use of GenAI

**Privacy — the one that gets people fired.** Anything you paste into a public AI tool may leave your organisation. Never paste customer data, credentials, medical records, or unreleased company code into a consumer chatbot.

**Verify before you ship.** You are responsible for what you submit, not the model. "The AI wrote it" is not a defence for a wrong number in a client report.

**Be honest about AI use.** Follow your institution's or employer's policy on disclosure. Undeclared AI use in assessed work is usually academic misconduct.

**Watch over-reliance.** Using an LLM to *explain* a concept builds skill. Using it to *replace* the thinking removes the very practice that makes you employable. In this course: use it to check and to learn, not to hand in.

**Bias and fairness.** If output affects real people — hiring, lending, grading — it needs auditing, not trust.

**Attribution and copyright.** Generated text and images sit in a genuinely unsettled legal area. Check before commercial use.

> A simple test before pasting anything into an AI tool:
> **"Would I be comfortable if this text appeared publicly, and if my name were attached to whatever comes back?"**

## 6. AI workflows — prompt chaining

One prompt rarely solves a real task. A **workflow** breaks the job into steps, where each step's output feeds the next.

```
   raw text
      │
      ▼
  [ STEP 1 ]  summarise
      │
      ▼
  [ STEP 2 ]  extract the complaints
      │
      ▼
  [ STEP 3 ]  classify sentiment
      │
      ▼
  structured result
```

Each step gets a small, clear instruction — which is exactly what LLMs handle best. Let's build one.

In [33]:
review = """I bought this laptop for machine learning work. The GPU is fast and
training runs much quicker than my old machine. However the battery only lasts
3 hours and the fan is very loud. Customer support took a week to reply."""

# --- step 1: summarise
summary = ask("Summarise this review in one sentence:\n\n" + review, 45)
print("STEP 1 — summary:\n ", summary, "\n")

# --- step 2: extract complaints
complaints = ask("List only the complaints in this review as short bullet points:\n\n" + summary, 70)
print("STEP 2 — complaints:\n", complaints, "\n")

# --- step 3: classify
sentiment = ask("Classify this review as exactly one word - positive, negative, or mixed:\n\n" + complaints, 8)
print("STEP 3 — sentiment:", sentiment)

STEP 1 — summary:
  The reviewer found their new laptop's GPU was faster and training ran more quickly compared to their previous machine, but the battery lasted only 3 hours and the fan was too loud, leading to customer support taking longer than expected. 

STEP 2 — complaints:
 - Faster GPU performance
- Faster training speed
- Battery lasts only 3 hours
- Loud fan noise
- Customer support takes longer than expected 

STEP 3 — sentiment: Negative


> Look closely at step 2 — a small model often slips a *positive* point into the complaints list. That's the lesson: **verify every step of a chain.** An error in step 1 silently corrupts everything downstream.

**Where workflows are used:** support-ticket triage, document summarisation pipelines, content generation with review stages, data extraction from forms.

## 7. Agentic workflows — the concept

A **chain** is fixed: you decide the steps in advance. An **agent** decides its own steps — it's given a goal and a set of **tools**, and it loops until it's done.

```
        ┌─────────────────────────────────────┐
        │                                     │
        ▼                                     │
   ┌─────────┐    ┌────────┐    ┌──────────┐  │
   │  THINK  │───►│   ACT  │───►│ OBSERVE  │──┘
   │ what do │    │ use a  │    │ read the │
   │ I do    │    │ tool   │    │ result   │
   │ next?   │    │        │    │          │
   └─────────┘    └────────┘    └──────────┘
        ▲                             │
        │      goal not reached       │
        └─────────────────────────────┘
                                      │ goal reached
                                      ▼
                                   ANSWER
```

**Tools** are just functions the model may call — a calculator, a web search, a database query, a file reader. The model doesn't run them; it *asks* for them, your code runs them and hands back the result.

**Chain vs agent:**

| | Prompt chain | Agent |
|---|---|---|
| Steps | you fix them | it chooses |
| Predictable | yes | no |
| Cost | fixed | can spiral |
| Debugging | easy | hard |
| Best for | known, repeatable tasks | open-ended tasks |

**Worked example — "What is the average temperature in Cairo this week, in Fahrenheit?"**
1. **Think:** I need current weather → I have a search tool.
2. **Act:** `search("Cairo weather this week")`
3. **Observe:** gets Celsius values.
4. **Think:** now I need to convert and average → I have a calculator.
5. **Act:** `calculate(...)`
6. **Observe:** gets the number.
7. **Think:** goal reached → answer.

Notice the model never did the arithmetic or knew the weather. It *orchestrated* tools that did.

> **When NOT to use an agent:** if you already know the steps, write a chain. Agents are slower, cost more, and fail in stranger ways. Reach for one only when the path genuinely can't be known ahead of time.

## ✅ Recap

- **Hallucination** is confident, fluent falsehood — a direct consequence of next-token prediction, not a fixable bug.
- Ask twice: **unstable facts signal invention.**
- Other limits: letters/counting, arithmetic, knowledge cutoff, context window, non-determinism, bias.
- **Grounding** (supplying the facts, RAG-style) is the strongest defence; also ask for uncertainty, check sources, use the right tool, keep a human in the loop.
- **Responsible use:** protect private data, verify before shipping, disclose AI use, avoid over-reliance.
- **Chains** are fixed pipelines you design; **agents** choose their own steps using tools. Prefer a chain when you know the path.

**Next (Lab):** hunt hallucinations, catalogue limitations, ground a model to fix it, build a working prompt chain, then design your own AI-assisted workflow. Open `LLM_Session2_Lab.ipynb`.

That completes the course — you can now use these models, and, more importantly, judge when not to. 🎉